In [56]:
import pandas as pd
import numpy as np
import spacy

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
df=pd.read_csv("C:/Users/mohan/Downloads/tamil_nadu_cm_qa.csv")
df.head(50)

,Question,Answer
0,In which year did P. S. Kumaraswamy Raja becom...,1949
1,In which year did P. S. Kumaraswamy Raja's ter...,1952
2,Which state did P. S. Kumaraswamy Raja govern ...,Madras State
3,In which year did C. Rajagopalachari become Ch...,1952
4,In which year did C. Rajagopalachari's term end?,1954
5,What earlier constitutional post did C. Rajago...,Governor-General of India
6,In which year did K. Kamaraj become Chief Mini...,1954
7,In which year did K. Kamaraj's term end?,1963
8,How many years did K. Kamaraj serve as Chief M...,9 years
9,What is K. Kamaraj widely known for besides be...,Kamaraj Plan


In [58]:
nlp = spacy.load("en_core_web_sm")

In [59]:
def clean_text(text):
    doc = nlp(str(text).lower())

    tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        tokens.append(token.lemma_)

    return " ".join(tokens)

df['Clean_Question'] = df['Question'].apply(clean_text)

print(df[['Question','Clean_Question']].head())

                                            Question  \
0  In which year did P. S. Kumaraswamy Raja becom...   
1  In which year did P. S. Kumaraswamy Raja's ter...   
2  Which state did P. S. Kumaraswamy Raja govern ...   
3  In which year did C. Rajagopalachari become Ch...   
4   In which year did C. Rajagopalachari's term end?   

                                      Clean_Question  
0  year p. s. kumaraswamy raja chief minister mad...  
1               year p. s. kumaraswamy raja term end  
2  state p. s. kumaraswamy raja govern rename tam...  
3  year c. rajagopalachari chief minister madras ...  
4                   year c. rajagopalachari term end  


In [60]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df['Clean_Question'])

X = tokenizer.texts_to_sequences(df['Clean_Question'])
X = pad_sequences(X, maxlen=12, padding='post')

In [61]:
vectorizer = TfidfVectorizer()
X_vectors = vectorizer.fit_transform(df['Clean_Question'])

print(X_vectors.shape)  


(47, 48)


In [62]:
le_ans = LabelEncoder()
y_ans = le_ans.fit_transform(df['Answer'])

In [63]:
X_train, X_test, y_ans_train, y_ans_test = train_test_split(X, y_ans, test_size=0.2, random_state=42)

In [64]:
vocab_size = len(tokenizer.word_index) + 1

model_ans = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=12),
    LSTM(128, return_sequences=True),
    LSTM(64),
    Dense(64, activation='relu'),
    Dense(len(le_ans.classes_), activation='softmax')
])

model_ans.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



early_stop = EarlyStopping(
    monitor='val_loss',         
    patience=3,                  
    restore_best_weights=True    
)


model_ans.fit(
    X_train, y_ans_train,
    epochs=50,
    validation_data=(X_test, y_ans_test))

C:\Users\mohan\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 936ms/step - accuracy: 0.0270 - loss: 3.5553 - val_accuracy: 0.0000e+00 - val_loss: 3.5610
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.0541 - loss: 3.5512 - val_accuracy: 0.0000e+00 - val_loss: 3.5659
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.0541 - loss: 3.5474 - val_accuracy: 0.0000e+00 - val_loss: 3.5706
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.0541 - loss: 3.5424 - val_accuracy: 0.0000e+00 - val_loss: 3.5760
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.0541 - loss: 3.5374 - val_accuracy: 0.0000e+00 - val_loss: 3.5836
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.0541 - loss: 3.5300 - val_accuracy: 0.0000e+00 - val_loss: 3.5937
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.0541 - loss: 3.5215 - val_accuracy: 0.0000e+00 - val_loss: 3.6058
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.0541 - loss: 3.5087 - val_accu

In [66]:
def predict_system(question, threshold=0.3):

    
    clean_q = clean_text(question)

    q_vector = vectorizer.transform([clean_q])

    similarities = cosine_similarity(q_vector, X_vectors)[0]

    best_idx = similarities.argmax()
    best_score = similarities[best_idx]

    if best_score < threshold:
        return "Sorry, I don't have an answer for that."

    return df['Answer'].iloc[best_idx]

In [67]:
print(predict_system("What was C. Joseph Vijay's profession before entering politics?"))

Film actor


In [68]:
print(predict_system("Which political party does C. Joseph Vijay lead as Chief Minister?"))

TVK


In [69]:
print(predict_system("What distinction does C. Joseph Vijay hold in Tamil Nadu's political history?"))

First TVK Chief Minister
